# 09 · Base-model benchmark scores and minimum detectable effect
Greedy decoding, 4096 tokens, answer from the last `\boxed{}`, graded with Math-Verify. Order: MATH-500 for both
models first (the primary benchmark), then the rest of the suite, then two sampled answers per MATH-500 problem to
measure how often two draws of the same model disagree, which sets the minimum detectable effect.
Runs resume: if the session drops, reconnect and run all again.
Results are written to the output folder as each run finishes.

In [ ]:
# Setup (safe to re-run): GPU check, code, dependencies, benchmark data, output folder.
import os, pathlib, subprocess, sys
USE_DRIVE = False       # True: keep outputs on Google Drive so a new session resumes them
BRANCH = "main"
REPO = pathlib.Path("/content/cmpe260-selfplay")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "NO GPU: Runtime > Change runtime type > T4 GPU")
if not REPO.exists():
    from google.colab import userdata  # repository is private: add a read-only GH_TOKEN in Colab Secrets
    url = f"https://{userdata.get('GH_TOKEN')}@github.com/Anuragbo/selfplay.git"
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)

os.chdir(REPO)

def sh(cmd, log="setup.log"):
    """Run a shell command; stdout to this cell, stderr to a log file whose tail is shown if it fails."""
    with open(log, "a") as err:
        p = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=err, text=True)
    print(p.stdout[-20000:])
    if p.returncode:
        print(f"FAILED ({p.returncode}): {cmd}\n--- last lines of {log} ---")
        print("".join(open(log).readlines()[-60:]))
        raise RuntimeError(cmd)

print(sys.version)
if not pathlib.Path(".deps_ok").exists():
    sh("pip install -q textarena==0.7.4 'math-verify[antlr4_13_2]==0.9.0'")
    try:  # vLLM is optional: without it the transformers backend is used
        sh("pip install -q vllm", log="vllm-install.log")
    except RuntimeError:
        print("vLLM did not install; the smoke test will fall back to the transformers backend")
    pathlib.Path(".deps_ok").touch()
if not pathlib.Path("data/raw/aime25.jsonl").exists():
    sh("bash data/fetch_benchmarks.sh")
OUT = pathlib.Path("/content/outputs")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = pathlib.Path("/content/drive/MyDrive/CMPE260_selfplay/outputs")
OUT.mkdir(parents=True, exist_ok=True)
sh('python -c "import sys; sys.path.insert(0, \".\"); from selfplay.generation import versions; print(versions())"')
sys.path.insert(0, str(REPO))
print("outputs ->", OUT)

In [ ]:
# Backend chosen by 00_smoke (vllm, vllm with VLLM_USE_V1=0, or hf). Change it here if needed.
BACKEND = pathlib.Path("backend.txt").read_text().strip() if pathlib.Path("backend.txt").exists() else "vllm"
if BACKEND == "vllm-v0":
    os.environ["VLLM_USE_V1"] = "0"
BACKEND_FLAG = "hf" if BACKEND == "hf" else "vllm"
print("backend:", BACKEND)

## MATH-500

In [ ]:
sh(f"python -m selfplay.evaluate --model Qwen/Qwen3-1.7B-Base --backend {BACKEND_FLAG} --benchmarks math500 "
   f"--temperature 0 --max-tokens 4096 --run-id eval-base-qwen3-1.7b --out-dir {OUT}/eval-base-qwen3-1.7b --print-summary",
   log="eval-base-qwen3-1.7b.log")

In [ ]:
sh(f"python -m selfplay.evaluate --model Qwen/Qwen3-4B-Base --backend {BACKEND_FLAG} --benchmarks math500 "
   f"--temperature 0 --max-tokens 4096 --run-id eval-base-qwen3-4b --out-dir {OUT}/eval-base-qwen3-4b --print-summary",
   log="eval-base-qwen3-4b.log")

## Rest of the suite (same output folders; MATH-500 is not regenerated)

In [ ]:
sh(f"python -m selfplay.evaluate --model Qwen/Qwen3-1.7B-Base --backend {BACKEND_FLAG} --benchmarks math500 olympiadbench minerva aime24 aime25 amc23 "
   f"--temperature 0 --max-tokens 4096 --run-id eval-base-qwen3-1.7b --out-dir {OUT}/eval-base-qwen3-1.7b --print-summary",
   log="eval-base-qwen3-1.7b.log")

In [ ]:
sh(f"python -m selfplay.evaluate --model Qwen/Qwen3-4B-Base --backend {BACKEND_FLAG} --benchmarks math500 olympiadbench minerva aime24 aime25 amc23 "
   f"--temperature 0 --max-tokens 4096 --run-id eval-base-qwen3-4b --out-dir {OUT}/eval-base-qwen3-4b --print-summary",
   log="eval-base-qwen3-4b.log")

## Two sampled answers per MATH-500 problem (1.7B), for the minimum detectable effect

In [ ]:
sh(f"python -m selfplay.evaluate --model Qwen/Qwen3-1.7B-Base --backend {BACKEND_FLAG} --benchmarks math500 "
   f"--samples 2 --temperature 0.6 --top-p 0.95 --max-tokens 4096 --run-id eval-base-qwen3-1.7b-t0.6x2 "
   f"--out-dir {OUT}/eval-base-qwen3-1.7b-t0.6x2 --print-summary", log="eval-sampled.log")

In [ ]:
# Minimum detectable effect from the sampled run (two draws of the same model).
import json
from selfplay.evaluate import _read_jsonl
from selfplay.stats import ci_half_width, minimum_detectable_effect, split_half_discordance
rows = _read_jsonl(OUT / "eval-base-qwen3-1.7b-t0.6x2" / "per_problem.jsonl")
by = {}
for r in rows:
    by.setdefault(r["problem_id"], {})[r["sample"]] = int(r["correct"])
samples = [[s[k] for k in sorted(s)] for s in by.values()]
d = split_half_discordance(samples)
print("MDE_BEGIN")
print(json.dumps({"problems": len(samples), "discordance": d,
                  "ci_half_width_math500": ci_half_width(d, len(samples)),
                  "mde80_math500": minimum_detectable_effect(d, len(samples)),
                  "ci_half_width_pooled_1547": ci_half_width(d, 1547),
                  "mde80_pooled_1547": minimum_detectable_effect(d, 1547)}))
print("MDE_END")